# Практическая работа к Лекции 3 — Transformers & LLMs
**Тема:** стратегии генерации, промпт‑инжиниринг, детерминированный JSON, скорость инференса.

**Цель:** закрепить теорию из лекции на практике.

## 0. Окружение
Установите зависимости и выберите модель (HF Transformers).

### Пояснения для студентов — Раздел 0 «Окружение»
- **Модель и память.** Если `TinyLlama-1.1B` не помещается в VRAM, замените на ещё более лёгкую (`Qwen2.5-0.5B-Instruct`) или выставьте `DTYPE="float16"` и `DEVICE_MAP="auto"`.
- **CPU‑режим.** При отсутствии GPU удалите `device_map` и передайте `torch_dtype=None` (будет медленнее, но стабильно).
- **Кэш HF.** Модели скачиваются в `~/.cache/huggingface/`. При первой загрузке это может занять время.
- **Проверка токенайзера.** Если видите предупреждение про `pad_token_id`, мы уже пробрасываем `pad_token_id=tokenizer.eos_token_id` — этого достаточно.
- **Стабильность результатов.** Мы фиксируем `SEED=7`. Для повторяемости не меняйте seed между прогонами сравнения.
- **Экономия времени.** В задачах со свипами (Задание 2) можно уменьшить сетку (`temps`, `top_p`, `top_k`) для быстрых черновиков, но в финальной версии оставьте хотя бы по 3 значения на параметр.
- **Типичные ошибки:** OOM → уменьшите размер модели/DTYPE/`max_new_tokens`; ImportError → выполните ячейку с `pip install` и перезапустите окружение.

In [ ]:
# !pip install -U transformers accelerate sentencepiece jsonschema pyyaml matplotlib

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # замените при необходимости
DEVICE_MAP = "auto"
DTYPE = "bfloat16"  # или "float16"
SEED = 7

import torch, random, numpy as np, time, json, re
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map=DEVICE_MAP, torch_dtype=getattr(torch, DTYPE, torch.bfloat16)
)

def chat_prompt(system, user):
    if hasattr(tokenizer, "apply_chat_template"):
        msgs=[{"role":"system","content":system},{"role":"user","content":user}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user

def generate(system, user, max_new_tokens=256, **dec):
    prompt = chat_prompt(system, user)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    out = model.generate(
        **enc,
        do_sample=dec.get("do_sample", False),
        temperature=dec.get("temperature", 0.0),
        top_p=dec.get("top_p", 1.0),
        top_k=dec.get("top_k", 0),
        repetition_penalty=dec.get("repetition_penalty", 1.0),
        no_repeat_ngram_size=dec.get("no_repeat_ngram_size", 0),
        max_new_tokens=max_new_tokens,
        use_cache=dec.get("use_cache", True),
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=[tokenizer.eos_token_id] if tokenizer.eos_token_id else None
    )
    dt = time.perf_counter() - t0
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[-1500:], dt

def distinct_n(text, n=2):
    toks = text.split()
    if len(toks) < n: return 0.0
    ngrams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    return len(set(ngrams)) / max(1, len(ngrams))

def repetition_ratio(text):
    toks = text.split()
    return 0.0 if not toks else Counter(toks).most_common(1)[0][1] / len(toks)

def extract_between(text, start="<<<JSON", end="JSON>>>"):
    if start in text and end in text:
        s = text.index(start) + len(start)
        e = text.index(end, s)
        return text[s:e].strip()
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0).strip() if m else None


## Задание 1. Сравнить стратегии декодирования (greedy / beam / sampling)

### Пояснения — Задание 1 «Стратегии декодирования»
**Зачем сравниваем:**
- *Greedy* даёт наиболее детерминированный и сдержанный ответ, но может «застревать» в штампах.
- *Beam search* повышает сходимость к «сильной» последовательности, но иногда усредняет стиль и снижает разнообразие.
- *Sampling* (temperature/top‑p/top‑k) повышает разнообразие — полезно для креатива и тонких формулировок.

**Метрики интерпретировать так:**
- `distinct-1/2` — доля уникальных 1‑ и 2‑грамм. Выше → разнообразнее формулировки.
- `repetition_ratio` — доля самого частого токена. Ниже → меньше тавтологии.

**Как оценивать:**
1) Смотрите **смысл** и точность объяснения KV‑кэша (не только метрики).
2) Сравните **время генерации** — beam обычно медленнее, sampling близок к greedy.
3) Зафиксируйте **итог: где вы применили бы каждую стратегию** (объяснения vs креатив).

**Подводные камни:**
- Beam с `early_stopping=True` может преждевременно завершать генерацию — это нормально для коротких ответов.
- Слишком высокая `temperature` (>1.0) часто ухудшает фактическую точность.
- Сравнение корректнее при **одинаковом промпте** и схожей длине ответов.

In [ ]:
SYSTEM = "Ты — краткий и точный технический ассистент."
USER = "Объясни простыми словами, что делает KV-кэш в трансформерах и почему он ускоряет генерацию."

# 1) Greedy
greedy_text, greedy_dt = generate(SYSTEM, USER, do_sample=False, temperature=0.0, top_p=1.0, top_k=0, max_new_tokens=220)
print("GREEDY time: %.3fs" % greedy_dt)
print(greedy_text)

# 2) Beam-search
prompt = chat_prompt(SYSTEM, USER)
enc = tokenizer(prompt, return_tensors="pt").to(model.device)
t0 = time.perf_counter()
beam_out = model.generate(**enc, num_beams=4, early_stopping=True, max_new_tokens=220, pad_token_id=tokenizer.eos_token_id)
beam_dt = time.perf_counter() - t0
beam_text = tokenizer.decode(beam_out[0], skip_special_tokens=True)[-1500:]
print("\nBEAM time: %.3fs" % beam_dt)
print(beam_text)

# 3) Sampling
samp_text, samp_dt = generate(SYSTEM, USER, do_sample=True, temperature=0.8, top_p=0.95, top_k=50, max_new_tokens=220)
print("\nSAMPLE time: %.3fs" % samp_dt)
print(samp_text)

def report(label, text):
    print(f"\n== {label} ==")
    print("distinct-1: %.3f" % distinct_n(text,1))
    print("distinct-2: %.3f" % distinct_n(text,2))
    print("repetition_ratio: %.3f" % repetition_ratio(text))

report("GREEDY", greedy_text)
report("BEAM", beam_text)
report("SAMPLE", samp_text)


## Задание 2. Sweep гиперпараметров (temperature × top_p × top_k) + графики

### Пояснения — Задание 2 «Sweep гиперпараметров»
**Цель:** понять влияние `temperature`, `top_p`, `top_k` на стиль и устойчивость вывода.

**Рекомендации по процедуре:**
- Меняйте **один параметр за раз**, остальные фиксируйте — так легче интерпретировать графики.
- Для честного сравнения держите одинаковыми: системную роль, сам prompt, `max_new_tokens`.
- Если GPU, перед измерением времени можно добавить `torch.cuda.synchronize()` до/после `generate()`.

**Как читать графики:**
- Рост `temperature` обычно ↑ `distinct` и ↑ вариативность, но может ↑ ошибки.
- `top_p≈0.9–0.95` часто даёт хороший баланс, `top_k`=50–100 стабилизирует форму.

**Что сдать:** таблицу `df.head()`, 3 графика (distinct‑1/2, repetition vs temperature), и **краткие выводы** о «рабочих» настройках под вашу задачу.

In [ ]:
import itertools, pandas as pd, matplotlib.pyplot as plt

SYSTEM = "Ты — технический ассистент и пишешь кратко."
USER = "Опиши разницу между greedy, beam search и nucleus sampling за 3–4 предложения."

temps = [0.2, 0.5, 0.8, 1.0]
topp  = [0.8, 0.9, 0.95]
topk  = [0, 40, 100]

rows = []
for T, P, K in itertools.product(temps, topp, topk):
    text, dt = generate(SYSTEM, USER, do_sample=True, temperature=T, top_p=P, top_k=K, max_new_tokens=180)
    rows.append({
        "temperature": T, "top_p": P, "top_k": K,
        "time_s": dt, "len": len(text.split()),
        "distinct1": distinct_n(text,1),
        "distinct2": distinct_n(text,2),
        "repetition": repetition_ratio(text)
    })

df = pd.DataFrame(rows).sort_values(["top_p","top_k","temperature"])
df.head(10)

# Графики по temperature при top_p=0.95, top_k=50
sub = df[(df["top_p"]==0.95) & (df["top_k"]==50)]
plt.figure(); plt.plot(sub["temperature"], sub["distinct1"], marker="o"); plt.title("distinct-1 vs temperature"); plt.xlabel("temperature"); plt.ylabel("distinct-1"); plt.show()
plt.figure(); plt.plot(sub["temperature"], sub["distinct2"], marker="o"); plt.title("distinct-2 vs temperature"); plt.xlabel("temperature"); plt.ylabel("distinct-2"); plt.show()
plt.figure(); plt.plot(sub["temperature"], sub["repetition"], marker="o"); plt.title("repetition vs temperature"); plt.xlabel("temperature"); plt.ylabel("repetition"); plt.show()


## Задание 3. Детерминированный JSON‑экстрактор + валидация jsonschema

### Пояснения — Задание 3 «Детерминированный JSON‑экстрактор»
**Ключевая идея:** добиться стабильного, **валидного** JSON без лишнего текста.

**Почему такие настройки:** `temperature=0`, `top_k=0`, `top_p=1.0` → минимальная стохастика. Маркеры `<<<JSON … JSON>>>` упрощают извлечение.

**Как повысить валидность:**
- В системной роли жёстко требовать «верни ТОЛЬКО JSON».
- Добавить «если поле неизвестно — верни null». Избегайте пустых строк.
- В схемах дат используем паттерн `YYYY-MM`/`YYYY-MM-DD` — следите за форматом.
- При ошибке в валидации: распечатайте `obj` и сообщение валидатора, исправьте промпт (например, допишите, что `salary` — объект или `null`).

**Дополнительно (опция):** можно запускать **двухшаговый** конвейер — сначала черновик JSON, затем «self‑repair» (модель исправляет JSON по сообщению валидатора).

In [ ]:
from jsonschema import validate

schema = {
  "type":"object","required":["job_title","company_name","location","employment_type","remote_policy","description"],
  "additionalProperties": False,
  "properties": {
    "job_title":{"type":"string"},
    "company_name":{"type":"string"},
    "location":{"type":"object","properties":{"city":{"type":["string","null"]},"region":{"type":["string","null"]},"country":{"type":["string","null"]}},"required":["city","region","country"],"additionalProperties":False},
    "employment_type":{"type":"string","enum":["full_time","part_time","contract","internship","temporary","unspecified"]},
    "remote_policy":{"type":"string","enum":["onsite","hybrid","remote","unspecified"]},
    "salary":{"type":["object","null"],"properties":{"min":{"type":["number","null"]},"max":{"type":["number","null"]},"currency":{"type":["string","null"]},"period":{"type":["string","null"]}},"additionalProperties":False},
    "description":{"type":"string"}
  }
}

SYSTEM = "Ты — детерминированный экстрактор вакансий. Верни ТОЛЬКО JSON по схеме."
job_text = """Компания «Ритейл-Про» ищет Руководителя отдела продаж (HoS).
Город: Тюмень, режим работы: офис/гибрид (3/2).
Занятость: полная. Опыт: от 3 лет в B2B, управление командой 5+ человек.
ЗП: от 180 000 до 230 000 руб. в месяц + бонусы, ДМС, компенсация мобильной связи.
Обязанности: построение воронки продаж, планирование, развитие регионов, отчётность в CRM.
Требования: навыки переговоров, аналитика, Excel/BI, готовность к командировкам до 20%.
Контакты: Анна Петрова, hr@retailpro.ru, +7 (912) 000-00-00.
"""

USER = f"""Ниже текст вакансии. Выведи ТОЛЬКО валидный JSON, без комментариев.
Начни с строки <<<JSON и закончи JSON>>>.

Текст:
{job_text}

Требования к выводу:
- Строго соответствуй схеме (см. ключи и типы).
- Пустые/отсутствующие значения — null.
- Зарплату передай как объект: {{min,max,currency,period}} или null.
<<<JSON
"""

text, _ = generate(SYSTEM, USER, do_sample=False, temperature=0.0, top_p=1.0, top_k=0, max_new_tokens=450)
json_str = extract_between(text, "<<<JSON", "JSON>>>")
obj = json.loads(json_str) if json_str else None
print(json.dumps(obj, ensure_ascii=False, indent=2))

validate(instance=obj, schema=schema)
print("\nJSON валиден по схеме.")


## Задание 4. Промпт‑инжиниринг (Context / Instructions / Input / Constraints) + few‑shot

### Пояснения — Задание 4 «Few‑shot и структура промпта»
**Структура:**
- *Context* — роль и область.
- *Instructions* — что извлечь, формат вывода, строгие правила.
- *Input* — сам текст.
- *Constraints* — маркеры, формат дат, `null` вместо пустоты, запрет комментариев.

**Почему few‑shot помогает:** модель копирует формат и стиль примеров → выше доля валидных JSON.

**Эксперименты:**
- Сравните валидность при `T=0` (строгость) vs `T=0.5, top_p=0.9, top_k=50` (гибкость). Иногда `T=0` даёт меньше ошибок формата, но хуже покрытие полей; few‑shot это компенсирует.

**Лайфхаки:**
- Делайте примеры **короткими и показательной структуры**.
- Сортируйте поля одинаково во всех примерах.
- Добавьте в *Constraints* «никаких пояснений вне JSON».

In [ ]:
FEWSHOT = [
    ("Компания Альфа ищет Аналитика. Город: Москва. Занятость: полная. Режим: офис.",
     {"job_title":"Аналитик","company_name":"Компания Альфа",
      "location":{"city":"Москва","region":None,"country":"Россия"},
      "employment_type":"full_time","remote_policy":"onsite","salary":None,"description":"."})
]

def build_user(job_text):
    ctx  = "Контекст: ты структурируешь текст вакансии в JSON."
    instr= "Инструкция: извлеки ключевые поля; верни только JSON; пустые значения — null."
    cons = "Ограничения: ключи строго по схеме; начни с <<<JSON и закончи JSON>>>."
    import json as _json
    shots = "\n\n".join([f"Пример входа:\n{j}\nПример выхода (JSON):\n{_json.dumps(o,ensure_ascii=False)}" for j,o in FEWSHOT])
    return f"{ctx}\n{instr}\n\n{shots}\n\nВход:\n{job_text}\n\n{cons}\n<<<JSON"

JOBS = [
    "Компания Бета ищет Дизайнера UI/UX. Город: Санкт-Петербург. Занятость: частичная. Режим: гибрид.",
    "ООО «ТехноГрад» нанимает Инженера-тестировщика. Город: Казань. Режим: удалёнка. Занятость: полная.",
    "Сеть магазинов «Вектор» ищет Руководителя склада. Город: Новосибирск. Режим: офис."
]

from jsonschema import validate
def pct_valid(dec):
    ok=0
    for j in JOBS:
        t,_=generate("Ты — детерминированный экстрактор вакансий.", build_user(j), max_new_tokens=450, **dec)
        s=extract_between(t,"<<<JSON","JSON>>>")
        try:
            o=json.loads(s)
            validate(instance=o, schema=schema); ok+=1
        except Exception:
            pass
    return ok, len(JOBS)

ok_det, n = pct_valid({"do_sample":False,"temperature":0.0,"top_p":1.0,"top_k":0})
ok_smp, _ = pct_valid({"do_sample":True,"temperature":0.5,"top_p":0.9,"top_k":50})
print(f"Валидных JSON (детерминированно): {ok_det}/{n}")
print(f"Валидных JSON (sampling): {ok_smp}/{n}")


## Задание 5. Chain‑of‑Thought (CoT) vs без CoT — сравнение точности

### Пояснения — Задание 5 «CoT vs без CoT»
**Зачем:** проверить, улучшает ли явное пошаговое рассуждение точность на простых задачах.

**Как сравнивать:** одинаковые задачи, одинаковые лимиты токенов; в CoT‑режиме просим «реши по шагам…, затем выведи только число».

**Интерпретация:** если CoT даёт ↑ точность — значит, задача выигрывает от явной декомпозиции. На совсем простых задачах разница может быть небольшая.

**Внимание:** вы не публикуете рассуждения, а только число — это ближе к рабочим ограничениям в продуктивных системах.

In [ ]:
QA = [
    ("У Маши было 7 яблок, она купила ещё 5 и съела 3. Сколько осталось?", 9),
    ("Сумма чисел 18 и 27, затем вычти 10. Чему равен результат?", 35),
    ("В коробке было 12 карандашей, 4 сломались, докупили 7. Сколько стало?", 15),
    ("Сколько будет 8*7 минус 30?", 26),
    ("Если у Пети было 50 рублей, он потратил 18 и нашёл 5. Сколько денег у него теперь?", 37)
]

def ask(q, cot=False):
    sys_ = "Ты решаешь арифметические задачи и отвечаешь только числом."
    if cot:
        usr = q + "\nПоясни решение по шагам, а в конце выведи только число в строке Ответ: <число>."
    else:
        usr = q + "\nВыведи только число."
    t,_ = generate(sys_, usr, do_sample=False, temperature=0.0, max_new_tokens=80)
    m = re.search(r"(\d+)", t)
    return int(m.group(1)) if m else None

def eval_set(cot=False):
    ok=0
    for q,a in QA:
        pred = ask(q, cot=cot)
        ok += 1 if pred==a else 0
    return ok, len(QA)

ok_no, n = eval_set(cot=False)
ok_cot, _ = eval_set(cot=True)
print(f"Без CoT: {ok_no}/{n}")
print(f"CoT: {ok_cot}/{n}")


## Задание 6. Скорость с/без KV‑кэша — токены/сек

### Пояснения — Задание 6 «Скорость с/без KV‑кэша»
**Что меряем:** время на длинную генерацию и производительность (токенов/сек). `use_cache=True` включает KV‑кэш на декоде.

**Методика:**
- Для GPU добавьте `torch.cuda.synchronize()` перед/после `generate()` для точности тайминга.
- Запустите 2–3 прогона и усредните — кэш/тепло‑ап влияет на первый замер.
- Фиксируйте одинаковые `max_new_tokens`, промпт, параметры декодирования.

**Ожидание:** `use_cache=True` обычно быстрее на длинных ответах, особенно на больших моделях.
**Вывод:** отметьте различие производительности и в каких задачах вы обязаны держать кэш включённым.

In [ ]:
SYSTEM = "Ты — преподаватель по LLM. Поясни подробно."
USER = "Расскажи, как работает Mixture-of-Experts (MoE) в LLM, зачем роутер и как лечат routing collapse."

for flag in [True, False]:
    text, dt = generate(SYSTEM, USER, max_new_tokens=600, use_cache=flag, do_sample=False, temperature=0.0)
    n_toks = len(tokenizer.encode(text))
    print(f"use_cache={flag}: {n_toks} токенов за {dt:.2f}с → {n_toks/max(dt,1e-6):.1f} ток/с")


---

# Шаблон отчёта студента
**ФИО:** …  
**Группа:** …  
**Дата:** …  
**Модель/версия:** …  
**Среда выполнения (GPU/CPU, VRAM):** …  
**Seed:** 7 (или свой)

## Краткие результаты
| Задание | Ключевые параметры | Метрики/наблюдения |
|---|---|---|
| 1. Стратегии декодирования | … | Distinct‑1/2=…, repetition=…, время=… |
| 2. Sweep гиперпараметров | … | Графики: …, оптимальные значения: … |
| 3. JSON‑экстрактор | T=0, top_k=0, top_p=1.0 | Валидность JSON: …/…, ошибки: … |
| 4. Few‑shot промпт | … | Валидность T=0: …/…; T=0.5: …/… |
| 5. CoT vs no‑CoT | … | Точность: CoT …/…, без CoT …/… |
| 6. KV‑кэш | use_cache=True/False | Ток/сек: … / … |

## Скриншоты/графики
Вставьте графики из задания 2 и при необходимости другие иллюстрации.

## Выводы (3–7 предложений)
- Настройки для строгого JSON‑экстрактора: …
- Настройки для «естественных» объяснений: …
- Настройки для креативного текста: …
- Замечания по скорости и KV‑кэшу: …

## Дополнительно (опционально)
- Трюки промпта/стоп‑последовательности/валидация.
- Особенности выбранной модели (контекст‑окно, RoPE, GQA и т. д.).

### Как заполнять шаблон отчёта
- **Краткие результаты** — таблица со сводными метриками и параметрами.
- **Скриншоты/графики** — вставьте графики из Задания 2.
- **Выводы (3–7 предложений)** — сформулируйте «best‑practice» под три класса задач: строгий JSON, объяснения, креатив. Укажите, что вы изменили бы при переходе на другую модель/железо.